# Tutorial: Using EHR Data as Donor-Level Input in `cellink`

The `cellink` package was originally developed for integrating single-cell measurements with donor-level **genetic data**, but its modular design allows you to swap in **any structured donor-level modality**. In this tutorial, we demonstrate how to use **electronic health records (EHR)** as the donor-level input, replacing genotypes. 

This unlocks a wide range of applications, for example using blood pressure, lab values, or medical history in combination with cell-level transcriptomics.

## Setup

EHR support needs [ehrdata](https://ehrdata.readthedocs.io) (>= 0.4), which comes with the `ehr` extra:

```bash
pip install "cellink-tools[ehr]"
```

In [1]:
import numpy as np
import pandas as pd
import anndata as ad
import ehrdata as ed
from cellink import DonorData

## Create Repeated EHR Measurements

We begin by simulating repeated clinical measurements for two patients across three visits. These measurements could represent time-varying vital signs or lab results.

In [2]:
patients = pd.DataFrame(
    {"patient_id": ["P001", "P002"], "birthdate": ["1980-01-01", "1975-05-15"], "gender": ["M", "F"]}
).set_index("patient_id")

clinical_parameters = pd.DataFrame(
    {
        "parameter_id": ["BP_Systolic", "BP_Diastolic"],
        "name": ["Systolic Blood Pressure", "Diastolic Blood Pressure"],
        "unit": ["mmHg", "mmHg"],
    }
).set_index("parameter_id")

visits = pd.DataFrame({"visit_number": ["1", "2", "3"], "visit_id": ["V001", "V002", "V003"]}).set_index("visit_number")

# patients × clinical parameters × visits
repeated_measurements = np.array(
    [
        [
            [120, 118, 121],
            [81, 80, 82],
        ],
        [
            [130, 135, 125],
            [84, 81, 80],
        ],
    ],
    dtype=float,
)

## Construct the `EHRData` Object
We wrap the patient information, clinical parameters, and repeated measurements into an `EHRData` object, which extends `AnnData` by a time axis: `obs` holds the patients, `var` the clinical parameters, and `tem` the visits. The three-dimensional measurements go into the layer `tem_data`, ehrdata's default for time series data. `EHRData` mirrors the structure expected by cellink, and will later be used in place of genetic data.

In [3]:
ehr = ed.EHRData(
    layers={"tem_data": repeated_measurements},
    obs=patients,
    var=clinical_parameters,
    tem=visits,
)

ehr.obs["donor_id"] = ["D0", "D1"]
ehr.obs.index = ehr.obs["donor_id"]
ehr

EHRData object with n_obs × n_vars × n_t = 2 × 2 × 3
    obs: 'birthdate', 'gender', 'donor_id'
    var: 'name', 'unit'
    tem: '1', '2', '3'
    layers: 'tem_data'
    shape of .tem_data: (2, 2, 3)

## Simulate Single-Cell RNA-seq Data
To demonstrate multimodal integration, we generate a synthetic single-cell RNA-seq dataset. Each cell is assigned to a donor and annotated with a predicted cell type. We filter to keep only CD8 Naive cells.

In [4]:
rng = np.random.default_rng(0)
n_cells = 200
n_genes = 100

X = rng.poisson(1.5, size=(n_cells, n_genes)).astype(np.float32)

cell_obs = pd.DataFrame(
    {
        "cell_id": [f"C{i}" for i in range(n_cells)],
        "donor_id": rng.choice(["D0", "D1"], size=n_cells),
        "predicted.celltype.l2": rng.choice(["CD8 Naive", "CD4 TCM"], size=n_cells),
    }
).set_index("cell_id")

gene_var = pd.DataFrame(index=[f"gene_{i}" for i in range(n_genes)])

adata = ad.AnnData(X=X, obs=cell_obs, var=gene_var)
adata = adata[adata.obs["predicted.celltype.l2"] == "CD8 Naive", :].copy()

## Combine EHR and Single-Cell Data via `DonorData`
We now use `DonorData` to merge the donor-level EHR data with the cell-level transcriptomics. This object enables unified access and is used throughout the cellink pipeline. The `G` side is displayed as an `EHRData`, including its time axis and the `tem` annotation.

In [5]:
dd = DonorData(G=ehr, C=adata).copy()
dd

╔═ DonorData(n_donors=2, n_cells_per_donor=[41-53], donor_id='donor_id') ════════════════════════════════╗
║ ┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓ ║
║ ┃ G (donors)                                         ┃ C (cells)                                     ┃ ║
║ ┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩ ║
║ │ EHRData object with n_obs × n_vars × n_t = 2 × 2 × │ AnnData object with n_obs × n_vars = 94 × 100 │ ║
║ │ 3                                                  │                                               │ ║
║ │     obs: 'birthdate', 'gender', 'donor_id'         │     obs: 'donor_id', 'predicted.celltype.l2'  │ ║
║ │     var: 'name', 'unit'                            │                                               │ ║
║ │     tem: 'visit_id'                                │                                               │ ║
║ │     layers: 'tem_data'           

## Shape and Subsetting

`dd.shape` lists the four axes that `dd[...]` indexes: donors, `G` variables, cells and `C` variables. The time axis of the `EHRData` is not one of them; it is available as `dd.G.n_t`, and `dd.G.shape` gives all three dimensions of `G`.

In [6]:
print("dd.shape:  ", dd.shape)
print("dd.G.shape:", dd.G.shape)
print("dd.G.n_t:  ", dd.G.n_t)

dd.shape:   (2, 2, 94, 100)
dd.G.shape: (2, 2, 3)
dd.G.n_t:   3


Subsetting donors or clinical parameters keeps the `EHRData` and all of its visits; the cells of removed donors are dropped from `C` accordingly.

In [7]:
sub = dd[["D1"], ["BP_Systolic"]]
print(type(sub.G).__name__, sub.G.shape, "|", sub.C.n_obs, "cells")
np.asarray(sub.G.layers["tem_data"])

EHRData (1, 1, 3) | 53 cells


array([[[130., 135., 125.]]])

## Summarise the Time Series per Donor

Most donor-level analyses expect one value per donor, so a common step is to summarise the repeated measurements over time, e.g. the mean blood pressure across visits. These summaries can be stored in `dd.G.obs` and used like any other donor-level covariate, for instance next to pseudobulked expression.

In [8]:
tem_data = dd.G.layers["tem_data"]  # donors × parameters × visits
for j, parameter in enumerate(dd.G.var_names):
    dd.G.obs[f"mean_{parameter}"] = tem_data[:, j, :].mean(axis=1)

dd.aggregate(key_added="PB_CD8_Naive", sync_var=True)
dd.G.obs

,birthdate,gender,donor_id,mean_BP_Systolic,mean_BP_Diastolic
donor_id,,,,,
D0,1980-01-01,M,D0,119.666667,81.000000
D1,1975-05-15,F,D1,130.000000,81.666667


> Note: writing a `DonorData` whose `G` is an `EHRData` (`dd.write_dd`) is not supported yet. Save `dd.G` with ehrdata (e.g. `ed.io.write_h5ed`) and `dd.C` with anndata separately.